In [ ]:
!pip install -q pycbc gwpy
!pip install -q jax==0.12.0.1 jaxlib==0.12.0.1
!pip install -q --no-deps ripplegw==0.5.0
!git clone -q --depth 1 https://github.com/santiqwertyy/cazador-ondas.git
!git -C cazador-ondas log -1 --oneline

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from pycbc.filter import match, matched_filter, sigma
from pycbc.types import FrequencySeries
from pycbc.waveform import get_fd_waveform

sys.path.append("cazador-ondas/src")
from datos import cargar, gps_evento
from filtro_adaptado import parecido, psd_en, serie_snr, snr_optimo, transformada
from plantillas import generador, taylorf2

In [ ]:
EVENTO = "GW150914"
DATOS = Path("datos")
DURACION = 128
DURACION_FFT = 4
DURACION_SEGMENTO = 32
INICIO_SEGMENTO = -24
F_MIN = 20
F_MAX = 1024
MASAS = (38.1, 33.0)
GIRO = 0.0
VENTANA = 0.5
MODELOS = ["IMRPhenomD", "IMRPhenomXAS"]
PUNTOS = [(6, 0.19, -0.2), (6, 0.22, 0.25), (15, 0.25, 0.0), (30, 0.22, 0.25), (45, 0.25, -0.2)]
PUNTOS_TAYLORF2 = [(1.2, 0.25), (6.0, 0.19), (30.0, 0.25)]

In [ ]:
DATOS.mkdir(exist_ok=True)
t0 = gps_evento(EVENTO, DATOS)
series = {ifo: cargar(ifo, EVENTO, DURACION, DATOS) for ifo in ["H1", "L1"]}
fs = int(series["H1"].sample_rate.value)
tiempos = series["H1"].times.value - t0
n = DURACION_SEGMENTO * fs
df = fs / n
indices = np.arange(int(F_MIN / df), int(F_MAX / df) + 1)
inicio = np.searchsorted(tiempos, INICIO_SEGMENTO)
t_segmento = tiempos[inicio] + np.arange(n) / fs
datos_f = {ifo: transformada(serie.value[inicio : inicio + n], fs, fs // 2) for ifo, serie in series.items()}
psd = {ifo: psd_en(np.arange(n // 2 + 1) * df, serie.value, fs, DURACION_FFT) for ifo, serie in series.items()}

In [ ]:
hp, _ = get_fd_waveform(approximant="IMRPhenomD", mass1=MASAS[0], mass2=MASAS[1], spin1z=GIRO, spin2z=GIRO, delta_f=df, f_lower=F_MIN)
hp.resize(n // 2 + 1)
plantilla = hp.numpy()[indices]

In [ ]:
mias = {}
suyas = {}
cerca = np.abs(t_segmento) < VENTANA
for ifo in ["H1", "L1"]:
    psd_pycbc = FrequencySeries(psd[ifo], delta_f=df)
    mias[ifo] = serie_snr(datos_f[ifo], plantilla[None, :], psd[ifo], df, n, indices)[0]
    suyas[ifo] = np.abs(matched_filter(hp, FrequencySeries(datos_f[ifo], delta_f=df), psd=psd_pycbc, low_frequency_cutoff=F_MIN, high_frequency_cutoff=F_MAX + df).numpy())
    a_2048 = serie_snr(datos_f[ifo], plantilla[None, :], psd[ifo], df, n // 2, indices)[0]
    i = np.argmax(mias[ifo] * cerca)
    j = np.argmax(suyas[ifo] * cerca)
    print(f"{ifo}: diferencia relativa máxima {np.abs(mias[ifo] - suyas[ifo]).max() / suyas[ifo].max():.1e}; SNR máximo mío {mias[ifo][i]:.4f} en t0 {t_segmento[i]:+.4f} s, PyCBC {suyas[ifo][j]:.4f} en t0 {t_segmento[j]:+.4f} s")
    print(f"    serie a 2048 Hz frente a las muestras pares de PyCBC: {np.abs(a_2048 - suyas[ifo][::2]).max() / suyas[ifo].max():.1e}")
    print(f"    σ: mío {snr_optimo(plantilla, psd[ifo], df, indices):.6e}, PyCBC {sigma(hp, psd=psd_pycbc, low_frequency_cutoff=F_MIN, high_frequency_cutoff=F_MAX + df):.6e}")

In [ ]:
fig, ejes = plt.subplots(2, 1, sharex=True)
for eje, ifo in zip(ejes, ["H1", "L1"]):
    eje.plot(t_segmento[cerca], mias[ifo][cerca], label="mía")
    eje.plot(t_segmento[cerca], suyas[ifo][cerca], "--", label="PyCBC")
    eje.set_ylabel("SNR")
    eje.set_title(ifo)
    eje.legend()
ejes[-1].set_xlabel("Tiempo desde t0 (s)");

In [ ]:
psd_pycbc = FrequencySeries(psd["H1"], delta_f=df)
for modelo in MODELOS:
    generar = generador(indices * df, F_MIN, modelo)
    for masa_chirp, eta, giro in PUNTOS:
        masa_total = masa_chirp / eta ** 0.6
        masa_1 = masa_total * (1 + np.sqrt(1 - 4 * eta)) / 2
        lal, _ = get_fd_waveform(approximant=modelo, mass1=masa_1, mass2=masa_total - masa_1, spin1z=giro, spin2z=giro, delta_f=df, f_lower=F_MIN)
        lal.resize(n // 2 + 1)
        mia = generar(masa_chirp, eta, giro)
        espectro = np.zeros(n // 2 + 1, dtype=complex)
        espectro[indices] = mia
        suyo = match(FrequencySeries(espectro, delta_f=df), lal, psd=psd_pycbc, low_frequency_cutoff=F_MIN, high_frequency_cutoff=F_MAX + df)[0]
        mio = parecido(mia[None, :], lal.numpy()[indices], psd["H1"], df, n, indices)[0]
        print(f"{modelo}, Mc {masa_chirp}, η {eta}, giro {giro:+}: parecido ripple–LALSuite {mio:.5f} (match de PyCBC {suyo:.5f})")

In [ ]:
for masa_chirp, eta in PUNTOS_TAYLORF2:
    masa_total = masa_chirp / eta ** 0.6
    masa_1 = masa_total * (1 + np.sqrt(1 - 4 * eta)) / 2
    lal, _ = get_fd_waveform(approximant="TaylorF2", mass1=masa_1, mass2=masa_total - masa_1, delta_f=df, f_lower=F_MIN, distance=1, phase_order=7, amplitude_order=0)
    lal.resize(n // 2 + 1)
    mia = taylorf2(indices * df, masa_chirp, eta)
    suya = lal.numpy()[indices]
    en_los_dos = (mia != 0) & (suya != 0)
    cociente = mia[en_los_dos] / suya[en_los_dos]
    desfase = np.unwrap(np.angle(cociente))
    recta = np.polyfit(indices[en_los_dos] * df, desfase, 1)
    resto = desfase - np.polyval(recta, indices[en_los_dos] * df)
    print(f"TaylorF2 a mano frente a LALSuite, Mc {masa_chirp}, η {eta}: hasta {indices[en_los_dos][-1] * df:.0f} Hz, amplitud − 1 hasta {np.abs(np.abs(cociente) - 1).max():.0e}, desfase sin constante ni tiempo {np.abs(resto).max():.0e} rad (corrimiento en tiempo {recta[0] / (2 * np.pi) * 1000:+.3f} ms)")